# Tech Challenge Fase 2  
## Notebook 01.4 — Bronze Metas Municípios

### Responsabilidade do notebook

Este notebook realiza a ingestão dos arquivos de **metas de alfabetização por município** para a camada **Bronze** do Data Lake.

A camada Bronze preserva os dados o mais próximo possível da origem, adicionando apenas metadados técnicos de rastreabilidade.

### Entrada

```text
raw/metas_municipios/
├── metas_municipios_2023.xlsx
├── metas_municipios_2024.xlsx
└── metas_municipios_2025.xlsx
```

### Saída

```text
bronze/metas_municipios/
├── ano=2023/
├── ano=2024/
└── ano=2025/
```

### Próximo notebook

```text
01_5_bronze_metas_ufs
```

## 1. Contexto na Arquitetura Medalhão

Este notebook atua exclusivamente na camada **Bronze**.

```text
Raw
 ↓
Bronze  ← você está aqui
 ↓
Silver
 ↓
Gold
```

A Bronze não aplica regras de negócio ou padronização analítica.  
Sua função é preservar as planilhas de metas municipais, mantendo rastreabilidade da origem e do ano de referência.

## 2. Particularidade das planilhas de metas municipais

Os arquivos de metas municipais são disponibilizados em formato Excel (`.xlsx`) e possuem duas linhas iniciais de cabeçalho:

- a primeira linha possui descrições amigáveis;
- a segunda linha contém os nomes técnicos das colunas.

Por isso, utilizamos `skiprows=1` na leitura com pandas, para que os nomes técnicos sejam utilizados como cabeçalho do DataFrame.

## 3. Importação das bibliotecas

Nesta etapa importamos as bibliotecas necessárias para:

- leitura do arquivo de configuração;
- leitura de arquivos Excel com pandas;
- manipulação de datas;
- criação de schemas explícitos;
- uso de funções Spark;
- geração de logs operacionais.

## 4. Pré-requisito técnico: openpyxl
O pandas utiliza a biblioteca openpyxl para leitura de arquivos .xlsx.

In [0]:
%pip install openpyxl

## 5. Reinicialização opcional do Python
Após instalar uma biblioteca com %pip, o Databricks pode solicitar reinicialização do Python.

Se necessário, descomente a linha abaixo e execute.

In [0]:
dbutils.library.restartPython()

## 6. Imports

In [0]:
import json
from datetime import datetime
import pandas as pd

from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    IntegerType,
    LongType
)

## 7. Leitura do arquivo de configuração

O projeto utiliza um arquivo central `config.json`, gerado no notebook de setup.

Essa abordagem evita caminhos fixos espalhados pelos notebooks e facilita mudanças futuras de ambiente, bucket, volume ou estrutura de diretórios.

In [0]:
CONFIG_FILE_PATH = "/Volumes/workspace/default/vol_trio_drive/projetos/fiap/tech_challenge_fase2/config/config.json"

config = json.loads(dbutils.fs.head(CONFIG_FILE_PATH))

BASE_PATH = config["environment"]["base_path"]
RAW_PATH = config["paths"]["raw_path"]
BRONZE_PATH = config["paths"]["bronze_path"]
LOG_PATH = config["paths"]["log_path"]
CONFIG_PATH = config["paths"]["config_path"]
EXECUTION_DATE = config["project"]["execution_date"]

print("BASE_PATH:", BASE_PATH)
print("RAW_PATH:", RAW_PATH)
print("BRONZE_PATH:", BRONZE_PATH)
print("LOG_PATH:", LOG_PATH)
print("CONFIG_PATH:", CONFIG_PATH)
print("EXECUTION_DATE:", EXECUTION_DATE)

## 8. Leitura dos metadados Bronze

O notebook orquestrador criou a tabela `bronze_metadata`, que contém as informações necessárias para processar cada arquivo.

Nesta etapa, filtramos apenas os registros referentes ao dataset `metas_municipios`.

In [0]:
metadata_path = f"{CONFIG_PATH}/bronze_metadata"

df_metadata = spark.read.parquet(metadata_path)

df_metadata_metas_municipios = (
    df_metadata
    .filter(F.col("dataset") == "metas_municipios")
    .orderBy("ano")
)

display(df_metadata_metas_municipios)

## 9. Validação dos metadados de metas municipais

Antes de iniciar a ingestão, validamos se a tabela de metadados contém os arquivos esperados para os anos de 2023, 2024 e 2025.

In [0]:
anos_esperados = [2023, 2024, 2025]

anos_metadata = [
    row["ano"] for row in df_metadata_metas_municipios.select("ano").distinct().collect()
]

anos_faltantes = sorted(list(set(anos_esperados) - set(anos_metadata)))

if anos_faltantes:
    raise Exception(f"Metadados ausentes para os anos: {anos_faltantes}")
else:
    print("Metadados de metas municipais encontrados para todos os anos esperados.")

## 10. Função para leitura de arquivos Excel

Arquivos Excel em Volumes do Databricks podem ser lidos diretamente pelo caminho `/Volumes/...`.

A função abaixo utiliza os parâmetros registrados na tabela de metadados:

- `file_path`: caminho do arquivo Excel;
- `sheet_name`: aba da planilha;
- `skip_rows`: quantidade de linhas a ignorar antes do cabeçalho técnico.

Após a leitura, removemos colunas e linhas completamente vazias.

In [0]:
def read_excel_bronze(file_path: str, sheet_name: str, skip_rows: int):
    df_pandas = pd.read_excel(
        file_path,
        sheet_name=sheet_name,
        skiprows=skip_rows,
        engine="openpyxl",
        dtype=str
    )

    df_pandas = df_pandas.dropna(axis=1, how="all")
    df_pandas = df_pandas.dropna(axis=0, how="all")

    df_pandas.columns = [
        str(col).strip()
        .replace(" ", "_")
        .replace(".", "_")
        .replace("-", "_")
        for col in df_pandas.columns
    ]

    df_pandas = df_pandas.fillna("")

    for col in df_pandas.columns:
        df_pandas[col] = df_pandas[col].astype(str).str.strip()

    # Cria Spark DataFrame com schema string explícito
    schema = StructType([
        StructField(str(col), StringType(), True)
        for col in df_pandas.columns
    ])

    df_spark = spark.createDataFrame(
        df_pandas.astype(str).values.tolist(),
        schema=schema
    )

    return df_spark

## 11. Função de enriquecimento técnico da Bronze

A Bronze preserva os dados originais, mas recebe metadados técnicos de rastreabilidade:

- dataset;
- arquivo de origem;
- formato;
- ano de referência;
- timestamp de ingestão;
- data de execução;
- etapa do pipeline.

In [0]:
def add_bronze_metadata(df, dataset, source_file, source_format, ano_referencia):
    return (
        df
        .withColumn("_dataset", F.lit(dataset))
        .withColumn("_source_file", F.lit(source_file))
        .withColumn("_source_format", F.lit(source_format))
        .withColumn("_ano_referencia", F.lit(int(ano_referencia)))
        .withColumn("_ingestion_timestamp", F.current_timestamp())
        .withColumn("_execution_date", F.lit(EXECUTION_DATE))
        .withColumn("_pipeline_step", F.lit("bronze_metas_municipios"))
    )

## 12. Execução da ingestão Bronze — Metas Municípios

Nesta etapa o notebook percorre os metadados dos arquivos de metas municipais, lê cada Excel, adiciona metadados técnicos e grava em Parquet na camada Bronze.

A saída é organizada por ano:

```text
bronze/metas_municipios/ano=2023
bronze/metas_municipios/ano=2024
bronze/metas_municipios/ano=2025
```

Como as planilhas são pequenas, utilizamos `coalesce(1)` para reduzir fragmentação de arquivos.

In [0]:
execution_logs = []

rows_metadata = df_metadata_metas_municipios.collect()

for row in rows_metadata:
    dataset = row["dataset"]
    ano = int(row["ano"])
    file_name = row["file_name"]
    raw_path = row["raw_path"]
    bronze_output_path = row["bronze_path"]
    source_format = row["source_format"]
    sheet_name = row["sheet_name"]
    skip_rows = int(row["skip_rows"]) if row["skip_rows"] is not None else 0

    start_time = datetime.now()

    print("=" * 100)
    print(f"Iniciando ingestão Bronze — Dataset: {dataset} | Ano: {ano}")
    print(f"Arquivo origem: {raw_path}")
    print(f"Aba Excel: {sheet_name}")
    print(f"Destino Bronze: {bronze_output_path}")

    try:
        df_raw = read_excel_bronze(
            file_path=raw_path,
            sheet_name=sheet_name,
            skip_rows=skip_rows
        )

        df_bronze = add_bronze_metadata(
            df=df_raw,
            dataset=dataset,
            source_file=raw_path,
            source_format=source_format,
            ano_referencia=ano
        )

        records_read = df_bronze.count()
        columns_count = len(df_bronze.columns)

        (
            df_bronze
            .coalesce(1)
            .write
            .mode("overwrite")
            .format("parquet")
            .option("compression", "snappy")
            .save(bronze_output_path)
        )

        end_time = datetime.now()

        execution_logs.append({
            "dataset": str(dataset),
            "ano": int(ano),
            "file_name": str(file_name),
            "source_path": str(raw_path),
            "target_path": str(bronze_output_path),
            "status": "SUCCESS",
            "records_read": int(records_read),
            "columns_count": int(columns_count),
            "start_time": start_time.isoformat(),
            "end_time": end_time.isoformat(),
            "error_message": ""
        })

        print(f"Ingestão concluída com sucesso para {dataset} {ano}")

    except Exception as e:
        end_time = datetime.now()

        execution_logs.append({
            "dataset": str(dataset),
            "ano": int(ano),
            "file_name": str(file_name),
            "source_path": str(raw_path),
            "target_path": str(bronze_output_path),
            "status": "FAILED",
            "records_read": 0,
            "columns_count": 0,
            "start_time": start_time.isoformat(),
            "end_time": end_time.isoformat(),
            "error_message": str(e)
        })

        print(f"Erro na ingestão do arquivo {file_name}: {e}")

## 13. Criação do log de execução

Após a ingestão, os resultados são convertidos em DataFrame Spark com schema explícito, evitando problemas de inferência de tipos.

In [0]:
schema_execution_logs = StructType([
    StructField("dataset", StringType(), True),
    StructField("ano", IntegerType(), True),
    StructField("file_name", StringType(), True),
    StructField("source_path", StringType(), True),
    StructField("target_path", StringType(), True),
    StructField("status", StringType(), True),
    StructField("records_read", LongType(), True),
    StructField("columns_count", IntegerType(), True),
    StructField("start_time", StringType(), True),
    StructField("end_time", StringType(), True),
    StructField("error_message", StringType(), True)
])

df_execution_logs = spark.createDataFrame(
    execution_logs,
    schema=schema_execution_logs
)

display(df_execution_logs.orderBy("ano"))

## 14. Persistência do log de execução

Os logs são armazenados em `logs/pipeline_execution/bronze`, permitindo auditoria posterior da execução.

In [0]:
log_output_path = f"{LOG_PATH}/pipeline_execution/bronze/metas_municipios_execution_date={EXECUTION_DATE}"

(
    df_execution_logs
    .coalesce(1)
    .write
    .mode("overwrite")
    .format("parquet")
    .option("compression", "snappy")
    .save(log_output_path)
)

print("Log de execução salvo em:")
print(log_output_path)

## 15. Validação da camada Bronze gerada

Após a gravação, lemos os arquivos Parquet gerados para confirmar se a ingestão foi concluída corretamente.

In [0]:
for row in rows_metadata:
    ano = int(row["ano"])
    bronze_output_path = row["bronze_path"]

    print("=" * 100)
    print(f"Validação Bronze — metas_municipios | ano={ano}")
    print(bronze_output_path)

    try:
        df_validacao = spark.read.parquet(bronze_output_path)
        print("Colunas:", len(df_validacao.columns))
        display(df_validacao.limit(5))
    except Exception as e:
        print(f"Erro ao validar Bronze metas_municipios ano={ano}: {e}")

## 16. Checklist final do notebook

Caso exista alguma falha, o notebook retorna uma exceção para impedir que os próximos passos sejam executados sobre uma Bronze incompleta.

In [0]:
falhas = df_execution_logs.filter(F.col("status") == "FAILED").count()

if falhas > 0:
    display(df_execution_logs.filter(F.col("status") == "FAILED"))
    raise Exception(f"Foram encontradas {falhas} falhas na ingestão Bronze de metas municipais.")
else:
    print("Checklist final concluído com sucesso.")
    print("Todos os arquivos de metas municipais foram ingeridos na camada Bronze.")

## Resultado esperado

Ao final deste notebook, espera-se que a camada Bronze de metas municipais esteja criada:

```text
bronze/metas_municipios/
├── ano=2023/
├── ano=2024/
└── ano=2025/
```

Cada pasta deve conter arquivos Parquet com os dados originais acrescidos de metadados técnicos.

---

## Próximo notebook

```text
01_5_bronze_metas_ufs
```